In [1]:
# Import relevant packages
import os
import ast
import gc
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import numpy as np
import pandas as pd
from transformers import AutoModel, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

os.makedirs('BestModel', exist_ok=True)

Device: cuda


In [2]:
# Load dataset
print("\n1. Loading dontpatronizeme_pcl.tsv...")

# Skip first 4 lines (disclaimer)
pcl_df = pd.read_csv(
    'dontpatronizeme_pcl.tsv',
    sep='\t',
    skiprows=4,
    names=['par_id', 'art_id', 'keyword', 'country_code', 'text', 'orig_label']
)
pcl_df['text'] = pcl_df['text'].fillna('').astype(str)
pcl_df['par_id'] = pcl_df['par_id'].astype(str) 

print(f"   Loaded {len(pcl_df)} paragraphs")
print(f"   Columns: {pcl_df.columns.tolist()}")

print("\n2. Loading train.csv (Official train set)...")

train_labels_df = pd.read_csv('train.csv')
print(f"   Loaded {len(train_labels_df)} training labels")
print(f"   Columns: {train_labels_df.columns.tolist()}")
print(train_labels_df.head())

# Parse the label column (string representation of list to actual list)
def parse_label(label_str):
    """Convert string of label columns to binary label."""
    try:
        if isinstance(label_str, str):
            label_list = ast.literal_eval(label_str)
        else:
            label_list = label_str
        # Binary: 1 if any category is 1, else 0
        return 1 if any(label_list) else 0
    except:
        return 0

train_labels_df['binary_label'] = train_labels_df['label'].apply(parse_label)
train_labels_df['par_id'] = train_labels_df['par_id'].astype(str)

print(f"\n   Binary label distribution:")
print(f"   PCL (1): {(train_labels_df['binary_label'] == 1).sum()}")
print(f"   No PCL (0): {(train_labels_df['binary_label'] == 0).sum()}")

print("\n3. Loading dev.csv (Official Dev Set)...")

dev_labels_df = pd.read_csv('dev.csv')
print(f"   Loaded {len(dev_labels_df)} dev labels")
print(dev_labels_df.head())

dev_labels_df['binary_label'] = dev_labels_df['label'].apply(parse_label)
dev_labels_df['par_id'] = dev_labels_df['par_id'].astype(str)  # Keep as string

print(f"\n   Binary label distribution:")
print(f"   PCL (1): {(dev_labels_df['binary_label'] == 1).sum()}")
print(f"   No PCL (0): {(dev_labels_df['binary_label'] == 0).sum()}")

print("\n4. Loading task4_test.tsv (Official Test Set)...")

test_df = pd.read_csv(
    'task4_test.tsv',
    sep='\t',
    header=None,
    names=['par_id', 'art_id', 'keyword', 'country_code', 'text']
)
test_df['text'] = test_df['text'].fillna('').astype(str)
test_df['par_id'] = test_df['par_id'].astype(str)

print(f"   Loaded {len(test_df)} test samples")
print(f"   Columns: {test_df.columns.tolist()}")
print(test_df.head())


1. Loading dontpatronizeme_pcl.tsv...
   Loaded 10469 paragraphs
   Columns: ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'orig_label']

2. Loading train.csv (Official train set)...
   Loaded 8375 training labels
   Columns: ['par_id', 'label']
   par_id                  label
0    4341  [1, 0, 0, 1, 0, 0, 0]
1    4136  [0, 1, 0, 0, 0, 0, 0]
2   10352  [1, 0, 0, 0, 0, 1, 0]
3    8279  [0, 0, 0, 1, 0, 0, 0]
4    1164  [1, 0, 0, 1, 1, 1, 0]

   Binary label distribution:


   PCL (1): 794
   No PCL (0): 7581

3. Loading dev.csv (Official Dev Set)...
   Loaded 2094 dev labels
   par_id                  label
0    4046  [1, 0, 0, 1, 0, 0, 0]
1    1279  [0, 1, 0, 0, 0, 0, 0]
2    8330  [0, 0, 1, 0, 0, 0, 0]
3    4063  [1, 0, 0, 1, 1, 1, 0]
4    4089  [1, 0, 0, 0, 0, 0, 0]

   Binary label distribution:
   PCL (1): 199
   No PCL (0): 1895

4. Loading task4_test.tsv (Official Test Set)...


   Loaded 3832 test samples
   Columns: ['par_id', 'art_id', 'keyword', 'country_code', 'text']
  par_id      art_id     keyword country_code  \
0    t_0   @@7258997  vulnerable           us   
1    t_1  @@16397324       women           pk   
2    t_2  @@16257812     migrant           ca   
3    t_3   @@3509652     migrant           gb   
4    t_4    @@477506  vulnerable           ca   

                                                text  
0  In the meantime , conservatives are working to...  
1  In most poor households with no education chil...  
2  The real question is not whether immigration i...  
3  In total , the country 's immigrant population...  
4  Members of the church , which is part of Ken C...  


In [3]:
# Merge Training Data with paragraph data
print("\n1. Creating Training Dataset...")

train_df = train_labels_df.merge(
    pcl_df[['par_id', 'art_id', 'keyword', 'country_code', 'text']],
    on='par_id',
    how='left'
)

# Check for missing text
missing_text = train_df['text'].isna().sum()
if missing_text > 0:
    print(f"   Warning: {missing_text} samples have missing text")
    train_df = train_df.dropna(subset=['text'])

train_df = train_df[train_df['text'].str.strip().str.len() > 0].reset_index(drop=True)

print(f"   Training set: {len(train_df)} samples")
print(f"   PCL: {(train_df['binary_label'] == 1).sum()} ({(train_df['binary_label'] == 1).mean()*100:.1f}%)")
print(f"   No PCL: {(train_df['binary_label'] == 0).sum()} ({(train_df['binary_label'] == 0).mean()*100:.1f}%)")

# Merge Official Dev Data with paragraph data
print("\n2. Creating Official Dev Dataset...")

official_dev_df = dev_labels_df.merge(
    pcl_df[['par_id', 'art_id', 'keyword', 'country_code', 'text']],
    on='par_id',
    how='left'
)

# Check for missing text
missing_text = official_dev_df['text'].isna().sum()
if missing_text > 0:
    print(f"   Warning: {missing_text} samples have missing text")
    official_dev_df = official_dev_df.dropna(subset=['text'])

official_dev_df = official_dev_df[official_dev_df['text'].str.strip().str.len() > 0].reset_index(drop=True)

print(f"   Official Dev set: {len(official_dev_df)} samples")
print(f"   PCL: {(official_dev_df['binary_label'] == 1).sum()} ({(official_dev_df['binary_label'] == 1).mean()*100:.1f}%)")
print(f"   No PCL: {(official_dev_df['binary_label'] == 0).sum()} ({(official_dev_df['binary_label'] == 0).mean()*100:.1f}%)")

# Official Test Data
print("\n3. Official Test Dataset...")
print(f"   Official Test set: {len(test_df)} samples (no labels)")
print(f"   Sample par_ids: {test_df['par_id'].head().tolist()}")


print("\nData Loaded...")
print(f"Training set:     {len(train_df)} samples")
print(f"Official Dev set: {len(official_dev_df)} samples (used for local evaluation)")
print(f"Official Test set: {len(test_df)} samples (no labels, for submission)")


1. Creating Training Dataset...
   Training set: 8375 samples
   PCL: 794 (9.5%)
   No PCL: 7581 (90.5%)

2. Creating Official Dev Dataset...
   Official Dev set: 2093 samples
   PCL: 199 (9.5%)
   No PCL: 1894 (90.5%)

3. Official Test Dataset...
   Official Test set: 3832 samples (no labels)
   Sample par_ids: ['t_0', 't_1', 't_2', 't_3', 't_4']

Data Loaded...
Training set:     8375 samples
Official Dev set: 2093 samples (used for local evaluation)
Official Test set: 3832 samples (no labels, for submission)


In [4]:
# Contrastive Dataset

class ContrastiveDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128, num_pairs_per_sample=20):
        self.texts = np.array(texts)
        self.labels = np.array(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        self.pcl_idx = np.where(self.labels == 1)[0]
        self.no_pcl_idx = np.where(self.labels == 0)[0]
        
        self.pairs = self._generate_pairs(num_pairs_per_sample)
        print(f"Generated {len(self.pairs)} contrastive pairs")
    
    def _generate_pairs(self, num_pairs_per_sample):
        pairs = []
        
        for idx in self.pcl_idx:
            for _ in range(num_pairs_per_sample // 2):
                pos_idx = random.choice(self.pcl_idx)
                while pos_idx == idx:
                    pos_idx = random.choice(self.pcl_idx)
                pairs.append((idx, pos_idx, 1.0))
            
            for _ in range(num_pairs_per_sample // 2):
                neg_idx = random.choice(self.no_pcl_idx)
                pairs.append((idx, neg_idx, 0.0))
        
        no_pcl_sample = np.random.choice(
            self.no_pcl_idx, size=min(len(self.pcl_idx), len(self.no_pcl_idx)), replace=False
        )
        
        for idx in no_pcl_sample:
            for _ in range(num_pairs_per_sample // 2):
                pos_idx = random.choice(self.no_pcl_idx)
                while pos_idx == idx:
                    pos_idx = random.choice(self.no_pcl_idx)
                pairs.append((idx, pos_idx, 1.0))
            
            for _ in range(num_pairs_per_sample // 2):
                neg_idx = random.choice(self.pcl_idx)
                pairs.append((idx, neg_idx, 0.0))
        
        random.shuffle(pairs)
        return pairs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        idx1, idx2, label = self.pairs[idx]
        
        encoding1 = self.tokenizer(
            str(self.texts[idx1]), truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        encoding2 = self.tokenizer(
            str(self.texts[idx2]), truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        
        return {
            'input_ids1': encoding1['input_ids'].squeeze(0),
            'attention_mask1': encoding1['attention_mask'].squeeze(0),
            'input_ids2': encoding2['input_ids'].squeeze(0),
            'attention_mask2': encoding2['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

In [5]:
# Sentence Encoder

class SentenceEncoder(nn.Module):
    def __init__(self, model_name="sentence-transformers/all-mpnet-base-v2"):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.transformer.config.hidden_size
    
    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = self.mean_pooling(outputs, attention_mask)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings
    
    def encode(self, texts, tokenizer, batch_size=32, max_length=128, show_progress=True):
        self.eval()
        all_embeddings = []
        
        iterator = range(0, len(texts), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Encoding")
        
        with torch.no_grad():
            for i in iterator:
                batch_texts = texts[i:i+batch_size]
                encoding = tokenizer(
                    batch_texts, truncation=True, padding=True,
                    max_length=max_length, return_tensors='pt'
                )
                input_ids = encoding['input_ids'].to(device)
                attention_mask = encoding['attention_mask'].to(device)
                embeddings = self.forward(input_ids, attention_mask)
                all_embeddings.append(embeddings.cpu().numpy())
        
        return np.vstack(all_embeddings)


class CosineLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.cos_sim = nn.CosineSimilarity(dim=1)
    
    def forward(self, embeddings1, embeddings2, labels):
        similarities = self.cos_sim(embeddings1, embeddings2)
        similarities = (similarities + 1) / 2
        return F.mse_loss(similarities, labels)

In [6]:
# Fine-tuning

def fine_tune(train_df, model_name="sentence-transformers/all-mpnet-base-v2",
                     num_pairs=20, epochs=2, batch_size=16, lr=2e-5):
    """Function to fine-tune the model"""
    
    print('Fine-tuning...')
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = SentenceEncoder(model_name)
    model = model.to(device)
    
    dataset = ContrastiveDataset(
        texts=train_df['text'].tolist(),
        labels=train_df['binary_label'].tolist(),
        tokenizer=tokenizer,
        max_length=128,
        num_pairs_per_sample=num_pairs
    )
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    criterion = CosineLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        progress = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in progress:
            input_ids1 = batch['input_ids1'].to(device)
            attention_mask1 = batch['attention_mask1'].to(device)
            input_ids2 = batch['input_ids2'].to(device)
            attention_mask2 = batch['attention_mask2'].to(device)
            labels = batch['label'].to(device)
            
            emb1 = model(input_ids1, attention_mask1)
            emb2 = model(input_ids2, attention_mask2)
            loss = criterion(emb1, emb2, labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            progress.set_postfix({'loss': f'{loss.item():.4f}'})
        
        print(f"Epoch {epoch+1} Loss: {epoch_loss/len(dataloader):.4f}")
    
    return model, tokenizer

# Fine-tune
model, tokenizer = fine_tune(train_df, epochs=2, num_pairs=20)

Fine-tuning...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated 31760 contrastive pairs



Epoch 1/2:   0%|                                                                                           | 0/1985 [00:00<?, ?it/s]

...


Epoch 2/2: 100%|██████████████████████████████████████████████████████████████████▉| 1984/1985 [04:16<00:00,  7.74it/s, loss=0.0000]


Epoch 2/2: 100%|██████████████████████████████████████████████████████████████████▉| 1984/1985 [04:16<00:00,  7.74it/s, loss=0.0001]


Epoch 2/2: 100%|███████████████████████████████████████████████████████████████████| 1985/1985 [04:16<00:00,  7.73it/s, loss=0.0001]


Epoch 2/2: 100%|███████████████████████████████████████████████████████████████████| 1985/1985 [04:16<00:00,  7.74it/s, loss=0.0001]

Epoch 2 Loss: 0.0087


In [7]:
# Generate Embeddings

print('Generating Embeddings...')

train_emb = model.encode(train_df['text'].tolist(), tokenizer)
dev_emb = model.encode(official_dev_df['text'].tolist(), tokenizer)
test_emb = model.encode(test_df['text'].tolist(), tokenizer)

print(f"Train: {train_emb.shape}, Dev: {dev_emb.shape}, Test: {test_emb.shape}")

Generating Embeddings...



Encoding:   0%|                                                                                             | 0/262 [00:00<?, ?it/s]


Encoding:   1%|▉                                                                                    | 3/262 [00:00<00:09, 27.92it/s]


Encoding:   2%|█▉                                                                                   | 6/262 [00:00<00:09, 27.11it/s]


Encoding:   3%|██▉                                                                                  | 9/262 [00:00<00:09, 27.39it/s]

...


Encoding:  95%|██████████████████████████████████████████████████████████████████████████████▊    | 114/120 [00:04<00:00, 27.62it/s]


Encoding:  98%|████████████████████████████████████████████████████████████████████████████████▉  | 117/120 [00:04<00:00, 27.32it/s]


Encoding: 100%|███████████████████████████████████████████████████████████████████████████████████| 120/120 [00:04<00:00, 27.85it/s]


Encoding: 100%|███████████████████████████████████████████████████████████████████████████████████| 120/120 [00:04<00:00, 27.83it/s]

Train: (8375, 768), Dev: (2093, 768), Test: (3832, 768)


In [8]:
# TF-IDF and PCL indicators

# TF-IDF
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.95)
tfidf.fit(train_df['text'].tolist())

tfidf_train = tfidf.transform(train_df['text'].tolist()).toarray()
tfidf_dev = tfidf.transform(official_dev_df['text'].tolist()).toarray()
tfidf_test = tfidf.transform(test_df['text'].tolist()).toarray()

# PCL indicators
def pcl_indicators(texts):
    patronizing = ['poor', 'unfortunate', 'helpless', 'needy', 'desperate', 'struggling', 'vulnerable']
    savior = ['save', 'rescue', 'help', 'support', 'donate', 'charity', 'give', 'provide']
    
    features = []
    for text in texts:
        text_lower = text.lower()
        words = text_lower.split()
        n = max(len(words), 1)
        
        f = [
            sum(1 for w in patronizing if w in text_lower),
            sum(1 for w in savior if w in text_lower),
            len(text),
            n,
            text.count('!'),
            sum(1 for w in ['we', 'us', 'our'] if w in words),
            sum(1 for w in ['they', 'them', 'their'] if w in words),
        ]
        features.append(f)
    return np.array(features, dtype=np.float32)

pcl_train = pcl_indicators(train_df['text'].tolist())
pcl_dev = pcl_indicators(official_dev_df['text'].tolist())
pcl_test = pcl_indicators(test_df['text'].tolist())

# Combine features
X_train = np.hstack([train_emb, tfidf_train, pcl_train])
X_dev = np.hstack([dev_emb, tfidf_dev, pcl_dev])
X_test = np.hstack([test_emb, tfidf_test, pcl_test])

print(f"Features: {X_train.shape[1]}")

Features: 3775


In [9]:
# Train classifiers

print('Training classifier...')

y_train = train_df['binary_label'].values
y_dev = official_dev_df['binary_label'].values

# Balance training data
pcl_idx = np.where(y_train == 1)[0]
no_pcl_idx = np.where(y_train == 0)[0]
n_sample = int(len(pcl_idx) * 1.5)
np.random.seed(42)
no_pcl_sampled = np.random.choice(no_pcl_idx, size=min(n_sample, len(no_pcl_idx)), replace=False)
balanced_idx = np.concatenate([pcl_idx, no_pcl_sampled])

X_train_bal = X_train[balanced_idx]
y_train_bal = y_train[balanced_idx]

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

# Train classifiers
classifiers = {
    'logistic': LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42),
    'rf': RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42),
}

dev_probs_list = []
test_probs_list = []

for name, clf in classifiers.items():
    print(f"Training {name}...")
    clf.fit(X_train_scaled, y_train_bal)
    dev_probs_list.append(clf.predict_proba(X_dev_scaled)[:, 1])
    test_probs_list.append(clf.predict_proba(X_test_scaled)[:, 1])

# Ensemble probabilities
dev_probs = np.mean(dev_probs_list, axis=0)
test_probs = np.mean(test_probs_list, axis=0)


Training classifier...


Training logistic...
Training rf...


In [10]:
# Save model to BestModel/
import pickle

print("Saving all model components...")

os.makedirs('BestModel', exist_ok=True)

# 1. Save the fine-tuned encoder weights
torch.save(model.state_dict(), 'BestModel/finetuned_encoder.pt')
print("Saved: finetuned_encoder.pt")

# 2. Save tokenizer
tokenizer.save_pretrained('BestModel/tokenizer')
print("Saved: tokenizer/")

# 3. Save all 4 classifiers
with open('BestModel/classifiers.pkl', 'wb') as f:
    pickle.dump(classifiers, f)
print("Saved: classifiers.pkl (logistic,rf)")

# 4. Save scaler
with open('BestModel/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved: scaler.pkl")

# 5. Save TF-IDF vectorizer
with open('BestModel/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("Saved: tfidf.pkl")

# 6. Save config (threshold, balance ratio, etc.)
model_config = {
    'approach': 'Fine-tuned Sentence Encoder + Logistic + RF Ensemble',
    'base_model': 'sentence-transformers/all-mpnet-base-v2',
    'threshold': 0.40,
    'balance_ratio': 1.5,
    'ensemble_classifiers': ['logistic', 'rf'],
    'all_classifiers': list(classifiers.keys()),
}
with open('BestModel/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Saved: model_config.json")

print("\nAll components saved to BestModel/")


Saving all model components...


Saved: finetuned_encoder.pt
Saved: tokenizer/
Saved: classifiers.pkl (logistic,rf)
Saved: scaler.pkl
Saved: tfidf.pkl
Saved: model_config.json

All components saved to BestModel/


In [ ]:
# Load saved model and perform predictions

import pickle

print("Loading saved model...")

# Load config
with open('BestModel/model_config.json', 'r') as f:
    loaded_config = json.load(f)
print(f"Config: {loaded_config}")

threshold = loaded_config['threshold']

# Load tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained('BestModel/tokenizer')
print("Loaded: tokenizer")

# Load fine-tuned encoder
loaded_model = SentenceEncoder(loaded_config['base_model'])
loaded_model.load_state_dict(torch.load('BestModel/finetuned_encoder.pt', map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()
print("Loaded: finetuned_encoder.pt")

# Load classifiers
with open('BestModel/classifiers.pkl', 'rb') as f:
    loaded_classifiers = pickle.load(f)
print(f"Loaded: classifiers ({list(loaded_classifiers.keys())})")

# Load scaler
with open('BestModel/scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
print("Loaded: scaler")

# Load TF-IDF
with open('BestModel/tfidf.pkl', 'rb') as f:
    loaded_tfidf = pickle.load(f)
print("Loaded: tfidf")



Loading saved model...
Config: {'approach': 'Fine-tuned Sentence Encoder + Logistic + RF Ensemble', 'base_model': 'sentence-transformers/all-mpnet-base-v2', 'threshold': 0.4, 'balance_ratio': 1.5, 'ensemble_classifiers': ['logistic', 'rf'], 'all_classifiers': ['logistic', 'rf']}


Loaded: tokenizer


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: finetuned_encoder.pt
Loaded: classifiers (['logistic', 'rf'])
Loaded: scaler


Loaded: tfidf


In [12]:
# Generate features for dev set and test set

# Encode texts
print("\nEncoding dev set...")
dev_emb_loaded = loaded_model.encode(official_dev_df['text'].tolist(), loaded_tokenizer)
print("Encoding test set...")
test_emb_loaded = loaded_model.encode(test_df['text'].tolist(), loaded_tokenizer)

# TF-IDF features
tfidf_dev_loaded = loaded_tfidf.transform(official_dev_df['text'].tolist()).toarray()
tfidf_test_loaded = loaded_tfidf.transform(test_df['text'].tolist()).toarray()

# PCL indicator features
pcl_dev_loaded = pcl_indicators(official_dev_df['text'].tolist())
pcl_test_loaded = pcl_indicators(test_df['text'].tolist())

# Combine features
X_dev_loaded = np.hstack([dev_emb_loaded, tfidf_dev_loaded, pcl_dev_loaded])
X_test_loaded = np.hstack([test_emb_loaded, tfidf_test_loaded, pcl_test_loaded])

# Scale
X_dev_scaled_loaded = loaded_scaler.transform(X_dev_loaded)
X_test_scaled_loaded = loaded_scaler.transform(X_test_loaded)

# Get predictions from logistic + rf ensemble
dev_probs_logistic = loaded_classifiers['logistic'].predict_proba(X_dev_scaled_loaded)[:, 1]
dev_probs_rf = loaded_classifiers['rf'].predict_proba(X_dev_scaled_loaded)[:, 1]
test_probs_logistic = loaded_classifiers['logistic'].predict_proba(X_test_scaled_loaded)[:, 1]
test_probs_rf = loaded_classifiers['rf'].predict_proba(X_test_scaled_loaded)[:, 1]

dev_probs_final = (dev_probs_logistic + dev_probs_rf) / 2
test_probs_final = (test_probs_logistic + test_probs_rf) / 2

# Apply threshold
dev_predictions = (dev_probs_final >= threshold).astype(int)
test_predictions = (test_probs_final >= threshold).astype(int)

print(f"\nDev predictions:  {dev_predictions.sum()} PCL / {len(dev_predictions)} total")
print(f"Test predictions: {test_predictions.sum()} PCL / {len(test_predictions)} total")



Encoding dev set...



Encoding:   0%|                                                                                              | 0/66 [00:00<?, ?it/s]


Encoding:   5%|███▉                                                                                  | 3/66 [00:00<00:02, 26.74it/s]


Encoding:   9%|███████▊                                                                              | 6/66 [00:00<00:02, 26.74it/s]


Encoding:  14%|███████████▋                                                                          | 9/66 [00:00<00:02, 27.26it/s]

...


Encoding:  94%|██████████████████████████████████████████████████████████████████████████████▏    | 113/120 [00:04<00:00, 27.78it/s]


Encoding:  97%|████████████████████████████████████████████████████████████████████████████████▏  | 116/120 [00:04<00:00, 27.13it/s]


Encoding:  99%|██████████████████████████████████████████████████████████████████████████████████▎| 119/120 [00:04<00:00, 26.96it/s]


Encoding: 100%|███████████████████████████████████████████████████████████████████████████████████| 120/120 [00:04<00:00, 27.88it/s]


Dev predictions:  205 PCL / 2093 total
Test predictions: 330 PCL / 3832 total


In [13]:
# Report results and save predictions

print("Predictions on official dev set...")

y_dev = official_dev_df['binary_label'].values

f1 = f1_score(y_dev, dev_predictions, pos_label=1)
precision = precision_score(y_dev, dev_predictions, pos_label=1)
recall = recall_score(y_dev, dev_predictions, pos_label=1)

print(f"Threshold: {threshold}")
print(f"F1:        {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

print("\n")
print(classification_report(y_dev, dev_predictions, target_names=['No PCL', 'PCL']))

print("Confusion Matrix:")
cm = confusion_matrix(y_dev, dev_predictions)
print(cm)

# Save predictions
print("\nSaving predictions...")

def save_predictions(predictions, filepath):
    with open(filepath, 'w') as f:
        for pred in predictions:
            f.write(f"{int(pred)}\n")
    print(f"Saved {len(predictions)} predictions to {filepath}")

save_predictions(dev_predictions, 'BestModel/dev.txt')
save_predictions(test_predictions, 'BestModel/test.txt')

# Save predicted probabilities
def save_probabilities(probs, filepath):
    with open(filepath, 'w') as f:
        for p in probs:
            f.write(f"{p:.6f}\n")
    print(f"Saved {len(probs)} probabilities to {filepath}")

save_probabilities(dev_probs_final, 'BestModel/dev_probs.txt')
save_probabilities(test_probs_final, 'BestModel/test_probs.txt')

# Save summary
perf_summary = f"""
Performance :
  Official Dev Set:
    F1:        {f1:.4f}
    Precision: {precision:.4f}
    Recall:    {recall:.4f}

Predictions:
  Dev:  {dev_predictions.sum()} PCL / {len(dev_predictions)} total
  Test: {test_predictions.sum()} PCL / {len(test_predictions)} total
{'='*70}
"""

print(perf_summary)
with open('BestModel/model_summary.txt', 'w') as f:
    f.write(perf_summary)

print("All files saved to BestModel/")


Predictions on official dev set...
Threshold: 0.4
F1:        0.6485
Precision: 0.6390
Recall:    0.6583


              precision    recall  f1-score   support

      No PCL       0.96      0.96      0.96      1894
         PCL       0.64      0.66      0.65       199

    accuracy                           0.93      2093
   macro avg       0.80      0.81      0.81      2093
weighted avg       0.93      0.93      0.93      2093

Confusion Matrix:
[[1820   74]
 [  68  131]]

Saving predictions...
Saved 2093 predictions to BestModel/dev.txt
Saved 3832 predictions to BestModel/test.txt
Saved 2093 probabilities to BestModel/dev_probs.txt
Saved 3832 probabilities to BestModel/test_probs.txt

Performance :
  Official Dev Set:
    F1:        0.6485
    Precision: 0.6390
    Recall:    0.6583

Predictions:
  Dev:  205 PCL / 2093 total
  Test: 330 PCL / 3832 total

All files saved to BestModel/
